In [2]:
import pandas as pd
from sklearn.neighbors import BallTree
import numpy as np
import requests
import time
import unicodedata

In [3]:
final_df = pd.read_csv("../data/final_df.csv")
sch_data = pd.read_csv("../outputs/Good_School_index.csv")

EARTH_RADIUS_KM = 6371

# Ensure year is int
final_df["year"] = final_df["year"].astype(int)
sch_data["year"] = sch_data["year"].astype(int)

# Convert HDB coords to radians
final_df["_lat_rad"] = np.radians(final_df["lat"])
final_df["_lon_rad"] = np.radians(final_df["lon"])

In [4]:
# lat lon for sch

SEARCH_URL = "https://www.onemap.gov.sg/api/common/elastic/search"

def geocode_onemap(search_val, session=None, auth_token=None, retries=3, sleep=0.3):
    s = session or requests.Session()
    headers = {"Authorization": auth_token} if auth_token else {}
    params = {
        "searchVal": search_val,
        "returnGeom": "Y",
        "getAddrDetails": "Y",
        "pageNum": 1
    }

    for i in range(retries):
        try:
            r = s.get(SEARCH_URL, params=params, headers=headers, timeout=15)
            r.raise_for_status()
            j = r.json()
            if int(j.get("found", 0)) > 0:
                best = j["results"][0]
                return float(best["LATITUDE"]), float(best["LONGITUDE"])
            return None, None
        except Exception:
            if i == retries - 1:
                return None, None
            time.sleep((i + 1) * sleep)



In [5]:
sess = requests.Session()

def normalize_name(s):
    if pd.isna(s):
        return s
    s = str(s).strip()
    s = unicodedata.normalize("NFKC", s)
    s = s.replace("’", "'").replace("‘", "'")
    return s

sch_data["search_school"] = sch_data["school"].map(normalize_name)

school_overrides = {
    "CHIJ (Toa Payoh)": "CHIJ Primary (Toa Payoh)",
    "St. Andrew's Junior": "St. Andrew's Junior School",
    "St. Anthony's": "St. Anthony's Primary School",
    "St. Anthony's Canossian": "St. Anthony's Canossian Primary School",
    "St. Gabriel's": "St. Gabriel's Primary School",
    "St. Hilda's": "St. Hilda's Primary School",
    "St. Joseph's Institution Junior": "St. Joseph's Institution Junior",
    "St. Margaret's": "St. Margaret's Primary School",
    "St. Stephen's": "St. Stephen's School",
    "CHIJ (Katong)": "CHIJ Katong Primary",
    "CHIJ Our Lady of Good Counsel": "CHIJ Our Lady of Good Counsel",
    "CHIJ Our Lady Queen of Peace": "CHIJ Our Lady Queen of Peace",
    "CHIJ (Kellock)": "CHIJ Kellock",
}

def geocode_school(name, session=None):
    # manual override first for known failures
    if name in school_overrides:
        queries = [school_overrides[name]]
    else:
        # generic rule for all others
        queries = [
            f"{name} Primary School",
            f"{name} School"
        ]
    
    for q in queries:
        lat, lon = geocode_onemap(q, session=session)
        if lat is not None and lon is not None:
            return lat, lon, q
    
    return None, None, None


sch_unique = sch_data["search_school"].dropna().unique()

sch_map = {}
sch_query_used = {}

for s in sch_unique:
    lat, lon, q_used = geocode_school(s, session=sess)
    sch_map[s] = (lat, lon)
    sch_query_used[s] = q_used

sch_data["lat"] = sch_data["search_school"].map(lambda x: sch_map.get(x, (None, None))[0])
sch_data["lon"] = sch_data["search_school"].map(lambda x: sch_map.get(x, (None, None))[1])
sch_data["query_used"] = sch_data["search_school"].map(sch_query_used)

In [6]:
print(sch_data.head())

   Unnamed: 0         school  year       GSI  good_school_75  good_school_80  \
0           0      Admiralty  2013  0.491796               0               0   
1           1  Ahmad Ibrahim  2013 -0.519466               0               0   
2           2        Ai Tong  2013  1.233564               1               1   
3           3      Alexandra  2013  0.430765               0               0   
4           4   Anchor Green  2013 -0.027182               0               0   

   good_school_85  good_school_90  search_school       lat         lon  \
0               0               0      Admiralty  1.442635  103.800040   
1               0               0  Ahmad Ibrahim  1.433153  103.832942   
2               1               1        Ai Tong  1.360583  103.833020   
3               0               0      Alexandra  1.291334  103.824425   
4               0               0   Anchor Green  1.390370  103.887165   

                     query_used  
0      Admiralty Primary School  
1  Ahm

In [8]:

# --------------------------------------------------
# Function: distance + nearest school name by year
# --------------------------------------------------
def nearest_school_by_year(hdb_df, sch_df, flag_col):
    n = len(hdb_df)
    
    dist_out = np.full(n, np.nan)
    name_out = np.array([None] * n, dtype=object)

    common_years = sorted(set(hdb_df["year"]) & set(sch_df["year"]))

    for yr in common_years:
        hdb_mask = hdb_df["year"] == yr
        sch_mask = (sch_df["year"] == yr) & (sch_df[flag_col] == 1)

        sch_subset = sch_df.loc[sch_mask, ["lat", "lon", "school"]].dropna()

        if sch_subset.empty:
            continue

        # Build BallTree
        school_coords = np.radians(sch_subset[["lat", "lon"]].values)
        tree = BallTree(school_coords, metric="haversine")

        # HDB subset
        hdb_idx = hdb_df.index[hdb_mask]
        hdb_coords = hdb_df.loc[hdb_idx, ["_lat_rad", "_lon_rad"]].values

        dist, ind = tree.query(hdb_coords, k=1)

        # Store distance
        dist_vals = dist.flatten() * EARTH_RADIUS_KM
        dist_out[hdb_df.index.get_indexer(hdb_idx)] = dist_vals

        # Store school names
        nearest_names = sch_subset.iloc[ind.flatten()]["school"].values
        name_out[hdb_df.index.get_indexer(hdb_idx)] = nearest_names

    return dist_out, name_out


# --------------------------------------------------
# Create variables
# --------------------------------------------------
d75, n75 = nearest_school_by_year(final_df, sch_data, "good_school_75")
d80, n80 = nearest_school_by_year(final_df, sch_data, "good_school_80")
d85, n85 = nearest_school_by_year(final_df, sch_data, "good_school_85")
d90, n90 = nearest_school_by_year(final_df, sch_data, "good_school_90")

final_df["dist_nearest_goodschool_75"] = d75
final_df["sch_name_75"] = n75

final_df["dist_nearest_goodschool_80"] = d80
final_df["sch_name_80"] = n80

final_df["dist_nearest_goodschool_85"] = d85
final_df["sch_name_85"] = n85

final_df["dist_nearest_goodschool_90"] = d90
final_df["sch_name_90"] = n90


# Cleanup
final_df.drop(columns=["_lat_rad", "_lon_rad"], inplace=True)

In [ ]:
print(final_df.head())


        month        town  flat_type block       street_name  floor_area_sqm  \
0  2013-01-01  ANG MO KIO          2   510  ANG MO KIO AVE 8            44.0   
1  2013-01-01  ANG MO KIO          2   314  ANG MO KIO AVE 3            44.0   
2  2013-01-01  ANG MO KIO          2   323  ANG MO KIO AVE 3            44.0   
3  2013-01-01  ANG MO KIO          3   170  ANG MO KIO AVE 4            61.0   
4  2013-01-01  ANG MO KIO          3   174  ANG MO KIO AVE 4            60.0   

  flat_model  lease_commence_date  resale_price  year  ...  d_0_1km_good90  \
0   Improved                 1980      253000.0  2013  ...               0   
1   Improved                 1978      270000.0  2013  ...               0   
2   Improved                 1977      283000.0  2013  ...               0   
3   Improved                 1986      305000.0  2013  ...               1   
4   Improved                 1986      320000.0  2013  ...               1   

   d_1_2km_good90 dist_nearest_goodschool_75      

In [10]:
final_df.to_csv("../data/hdb_nearest_sch.csv", index=False)